In [1]:
import json
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
import os
from pathlib import Path

root = Path().resolve().parent
os.chdir(root)
os.getcwd()

'C:\\Users\\Adesh\\OneDrive\\Documents\\ML NLP\\Projects\\03 End-to-End Evaluation and Benchmarking Framework for LLM Outputs and Retrieval-Based NLP Systems'

In [3]:
with open("data/dataset.json", "r") as f:
    processed_data = json.load(f)

In [4]:
import joblib

chunked_corpus = joblib.load("artifacts/chunked_corpus.pkl")
chunk_embeddings = joblib.load("artifacts/chunk_embeddings.pkl")

In [5]:
chunk_embeddings.shape

(256, 384)

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [8]:
def retrieve_top_k(query, k = 3):
    query_embedding = embedding_model.encode([query])
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    results = [chunked_corpus[idx] for idx in top_k_idx]
    return results

In [9]:
# checking
sample_query = processed_data[0]["question"]
results = retrieve_top_k(sample_query)
for res in results:
    print(res,"\n")

{'chunk_id': 0, 'source_id': '572817584b864d1900164463', 'text': 'Outward urban expansion is now prevented by the Metropolitan Green Belt, although the built-up area extends beyond the boundary in places, resulting in a separately defined Greater London Urban Area. Beyond this is the vast London commuter belt. Greater London is split for some purposes into Inner London and Outer London. The city is split by the River Thames into North and South, with an informal central London area in its interior. The coordinates of the nominal centre of London, traditionally considered to be the original Eleanor Cross at Charing Cross near the junction of Trafalgar Square and Whitehall, are'} 

{'chunk_id': 165, 'source_id': '57287a762ca10214002da3b8', 'text': "London is a major international air transport hub with the busiest city airspace in the world. Eight airports use the word London in their name, but most traffic passes through six of these. London Heathrow Airport, in Hillingdon, West London,

In [10]:
load_dotenv()
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [11]:
#baseline generative function
def gen_baseline_answer(question):
    prompt = f"""
    Answer the following question briefly and factually. If you are unsure, be honest and say "I don't know."
    Question: {question}"""

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": prompt}
        ],
        temperature = 0
    )
    
    return response.choices[0].message.content.strip()

In [12]:
#RAG function
def gen_rag_answer(question, k = 3):
    retrieved_chunks = retrieve_top_k(question, k = k)
    context = "\n".join([chunk["text"] for chunk in retrieved_chunks])

    prompt = f"""
    Use ONLY and ONLY the provided context to answer the question. If the answer cannot be found in the context, be honest and say "I don't know."
    Context:
    {context}
    Question: {question}"""

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": prompt}
        ],
        temperature = 0
    )

    return {
        "answer": response.choices[0].message.content.strip(),
        "retrieved_chunks": retrieved_chunks
    }

In [ ]:
#testing pipelines

sample = processed_data[0]
question = sample["question"]

baseline_answer = gen_baseline_answer(question)
rag_answer = gen_rag_answer(question)

print("Question: ", question)
print("Ground Truth Answer: ", sample["answer"])
print("Baseline Answer:", baseline_answer)
print("RAG Answer:", rag_answer["answer"])

Question:  Greater London is divided into what two groups of boroughs?
Ground Truth Answer:  ['Inner London and Outer London']
Baseline Answer: Greater London is divided into two groups of boroughs: the Inner London boroughs and the Outer London boroughs.
RAG Answer: Greater London is divided into Inner London and Outer London.


In [14]:
#subset eval
eval_subset = processed_data[:10]
eval_results = []

for i, sample in enumerate(eval_subset):
    question = sample["question"]
    ground_truth = sample["answer"]
    print(f"Samples processed: {i+1} out of 10")
    baseline_answer = gen_baseline_answer(question)
    rag_answer = gen_rag_answer(question)
    eval_results.append({
        "id": sample["id"],
        "question": question,
        "ground_truth": ground_truth,
        "baseline_answer": baseline_answer,
        "rag_answer": rag_answer["answer"],
        "retrieved_chunks": [chunk["text"] for chunk in rag_answer["retrieved_chunks"]]
    })

Samples processed: 1 out of 10
Samples processed: 2 out of 10
Samples processed: 3 out of 10
Samples processed: 4 out of 10
Samples processed: 5 out of 10
Samples processed: 6 out of 10
Samples processed: 7 out of 10
Samples processed: 8 out of 10
Samples processed: 9 out of 10
Samples processed: 10 out of 10


In [17]:
eval_results[2]

{'id': '56f9ed16f34c681400b0beef',
 'question': 'What did the Grand Alliance propose as the new standard for SDTV and HDTV?',
 'ground_truth': ['ATSC'],
 'baseline_answer': 'The Grand Alliance proposed the ATSC (Advanced Television Systems Committee) standards for digital television, which included specifications for both Standard Definition Television (SDTV) and High Definition Television (HDTV). The ATSC standard for HDTV defined resolutions of 720p, 1080i, and 1080p, while SDTV was defined with a resolution of 480i.',
 'rag_answer': 'The Grand Alliance proposed ATSC as the new standard for SDTV and HDTV.',
 'retrieved_chunks': ['DVB created first the standard for DVB-S digital satellite TV, DVB-C digital cable TV and DVB-T digital terrestrial TV. These broadcasting systems can be used for both SDTV and HDTV. In the US the Grand Alliance proposed ATSC as the new standard for SDTV and HDTV. Both ATSC and DVB were based on the MPEG-2 standard, although DVB systems may also be used to t

In [18]:
with open("artifacts/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent = 4)